<a href="https://colab.research.google.com/github/iftekharalamfahim/HFDRL-Hands-On/blob/main/TrainHuggyTheDog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bonus Unit 1: Train Huggy the Dog to fetch a stick

## Clone the ML-Agents Repository

In [ ]:
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

## Setup the Virtual Environment

In [ ]:
# check the version compatibility with ML-Agents
!python --version

In [ ]:
# Install Python 3.10 and venv packages via apt
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

In [ ]:
# Create an isolated virtual environment
!python3.10 -m venv /content/myenv

In [ ]:
# Patch python_requires in setup.py files to allow patch versions > 3.10.12
import re

def patch_setup(file_path):
    with open(file_path, "r") as f:
        content = f.read()
    # Replace <=3.10.12 with <=3.10.25 to allow current 3.10 patch releases
    content = re.sub(r'<=3\.10\.12', '<=3.10.25', content)
    with open(file_path, "w") as f:
        f.write(content)

patch_setup("/content/ml-agents/ml-agents-envs/setup.py")
patch_setup("/content/ml-agents/ml-agents/setup.py")

In [ ]:
# Upgrade pip and install ML-Agents into the virtual environment
!/content/myenv/bin/pip install --upgrade pip
!/content/myenv/bin/pip install -e /content/ml-agents/ml-agents-envs
!/content/myenv/bin/pip install -e /content/ml-agents/ml-agents

# 5. Connect the virtual environment's executables to this Colab session
import os
import sys

os.environ["PATH"] = "/content/myenv/bin:" + os.environ["PATH"]
sys.path.insert(0, "/content/myenv/lib/python3.10/site-packages")

In [ ]:
# Python Version in new Virtual Environment
!python --version

## Installing the dependencies

In [ ]:
%%capture
%cd ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

In [ ]:
# Download and move the environment zip file in ./trained-envs-executables/linux/
!mkdir ./trained-envs-executables
!mkdir ./trained-envs-executables/linux

In [ ]:
# Download the file Huggy.zip from https://github.com/huggingface/Huggy
!wget "https://github.com/huggingface/Huggy/raw/main/Huggy.zip" -O ./trained-envs-executables/linux/Huggy.zip

!unzip -q -o ./trained-envs-executables/linux/Huggy.zip -d ./trained-envs-executables/linux/

In [ ]:
# Make sure the file is accessible
!chmod -R 755 ./trained-envs-executables/linux/Huggy

## Check the Huggy Config File

In [ ]:
import os

# Create the directory path if it doesn't already exist
os.makedirs("/content/ml-agents/config/ppo", exist_ok=True)

# Define the configuration text
config_content = """behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000
"""

# Save the file to /content/ml-agents/config/ppo/Huggy.yaml
file_path = "/content/ml-agents/config/ppo/Huggy.yaml"
with open(file_path, "w") as f:
    f.write(config_content)

print(f"Successfully created configuration file at: {file_path}")

## Train the Agent

In [ ]:
%cd /content/ml-agents

!mlagents-learn ./config/ppo/Huggy.yaml \
  --env=./trained-envs-executables/linux/Huggy/Huggy \
  --run-id="Huggy" \
  --no-graphics

## View the Training Charts (TensorBoard)

In [ ]:
# Load the TensorBoard extension inside Colab
%load_ext tensorboard

# Launch TensorBoard pointing to the ML-Agents results folder
%tensorboard --logdir /content/ml-agents/results